In [ ]:
from pathlib import Path
from typing import Dict, List, Union
import pickle

from datasets import Dataset, DatasetDict, load_from_disk

from tqdm import tqdm
def load_ragbench_from_pickles(pickle_paths: List[Union[str, Path]]) -> Dict[str, DatasetDict]:
    """
    Загружает несколько .pkl/.pickle в единый словарь формата ragbench:
      {
        "<dataset_name>": DatasetDict({"train": Dataset, "validation": Dataset, "test": Dataset, ...}),
        ...
      }

    Поддерживаемые корневые объекты в каждом pickle:
      1) DatasetDict                              -> кладётся под именем файла (без расширения)
      2) dict{name: DatasetDict}                  -> вливается в общий словарь
      3) dict{name: {split: Dataset}}             -> приводится к DatasetDict и вливается

    Ничего не склеиваем и не меняем: просто читаем и собираем.
    При конфликте имён датасетов бросаем KeyError.
    """
    def _coerce_to_datasetdict(obj) -> DatasetDict:
        if isinstance(obj, DatasetDict):
            return obj
        if isinstance(obj, dict) and all(isinstance(v, Dataset) for v in obj.values()):
            return DatasetDict(obj) 
        raise TypeError("Object is not a DatasetDict nor {split: Dataset} dict.")

    big: Dict[str, DatasetDict] = {}

    for raw in tqdm(pickle_paths):
        p = Path(raw)

        with open(p, "rb") as f:
            obj = pickle.load(f)
        if isinstance(obj, DatasetDict):
            name = p.stem
            if name in big:
                raise KeyError(f"Duplicate dataset name '{name}' from file {p}")
            big[name] = obj
            continue
        if isinstance(obj, dict):
            for name, val in obj.items():
                dsd = _coerce_to_datasetdict(val)
                if name in big:
                    raise KeyError(f"Duplicate dataset name '{name}' inside {p}")
                big[str(name)] = dsd
            continue

        raise TypeError(f"Unsupported pickle root object in {p}: {type(obj)}")

    return big

In [18]:
new_dataset = load_ragbench_from_pickles(['../cuad_ru.pkl','../delucionqa_ru.pkl'])

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:00<00:00, 13.01it/s]


In [21]:
new_dataset

{'cuad': DatasetDict({
     train: Dataset({
         features: ['id', 'question', 'documents', 'response', 'generation_model_name', 'annotating_model_name', 'dataset_name', 'documents_sentences', 'response_sentences', 'sentence_support_information', 'unsupported_response_sentence_keys', 'adherence_score', 'overall_supported_explanation', 'relevance_explanation', 'all_relevant_sentence_keys', 'all_utilized_sentence_keys', 'trulens_groundedness', 'trulens_context_relevance', 'ragas_faithfulness', 'ragas_context_relevance', 'gpt3_adherence', 'gpt3_context_relevance', 'gpt35_utilization', 'relevance_score', 'utilization_score', 'completeness_score', 'question_ru', 'response_ru', 'documents_sentences_ru'],
         num_rows: 749
     })
     validation: Dataset({
         features: ['id', 'question', 'documents', 'response', 'generation_model_name', 'annotating_model_name', 'dataset_name', 'documents_sentences', 'response_sentences', 'sentence_support_information', 'unsupported_response_se

In [24]:
def sampled(bench, idx, dataset, fold, ru_field, field):
    return bench[dataset][fold][idx][ru_field], bench[dataset][fold][idx][field]
# sampled(new_dataset, 2, 'cuad', 'validation', 'documents_sentences_ru', 'documents_sentences')

In [26]:
from datasets import DatasetDict, concatenate_datasets
DatasetDict(new_dataset).save_to_disk("ragbench_ru")

Saving the dataset (1/1 shards): 100%|█████████████████████████████████████████████████████████████| 184/184 [00:00<00:00, 15768.38 examples/s]


In [52]:
from huggingface_hub import login
login(token='')

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: write).
Your token has been saved to /root/.cache/huggingface/token
Login successful


NameError: name 'json' is not defined

In [ ]:
correct_dataset = DatasetDict(new_dataset)

print("Правильная структура:")
for config_name, config_data in correct_dataset.items():
    print(f"{config_name}: {type(config_data)}")
    if hasattr(config_data, 'keys'):
        for split_name in config_data.keys():
            print(f"  - {split_name}: {type(config_data[split_name])}")

Правильная структура:
cuad: <class 'datasets.dataset_dict.DatasetDict'>
  - train: <class 'datasets.arrow_dataset.Dataset'>
  - validation: <class 'datasets.arrow_dataset.Dataset'>
  - test: <class 'datasets.arrow_dataset.Dataset'>
delucionqa: <class 'datasets.dataset_dict.DatasetDict'>
  - train: <class 'datasets.arrow_dataset.Dataset'>
  - validation: <class 'datasets.arrow_dataset.Dataset'>
  - test: <class 'datasets.arrow_dataset.Dataset'>


In [ ]:

DatasetDict(new_dataset['cuad']).push_to_hub(
    "CMCenjoyer/ragbench-ru",
    config_name="cuad",
    private=False, 
    commit_message="Add RAG-Bench Russian dataset with CUAD"
)

ValueError: All datasets in `DatasetDict` should have the same features but features for 'validation' and 'test' don't match: {'id': Value(dtype='string', id=None), 'question': Value(dtype='string', id=None), 'documents': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None), 'response': Value(dtype='string', id=None), 'generation_model_name': Value(dtype='string', id=None), 'annotating_model_name': Value(dtype='string', id=None), 'dataset_name': Value(dtype='string', id=None), 'documents_sentences': Sequence(feature=Sequence(feature=Sequence(feature=Value(dtype='string', id=None), length=-1, id=None), length=-1, id=None), length=-1, id=None), 'response_sentences': Sequence(feature=Sequence(feature=Value(dtype='string', id=None), length=-1, id=None), length=-1, id=None), 'sentence_support_information': [{'explanation': Value(dtype='string', id=None), 'fully_supported': Value(dtype='bool', id=None), 'response_sentence_key': Value(dtype='string', id=None), 'supporting_sent

In [37]:
DatasetDict(new_dataset['cuad'])

{'cuad': DatasetDict({
     train: Dataset({
         features: ['id', 'question', 'documents', 'response', 'generation_model_name', 'annotating_model_name', 'dataset_name', 'documents_sentences', 'response_sentences', 'sentence_support_information', 'unsupported_response_sentence_keys', 'adherence_score', 'overall_supported_explanation', 'relevance_explanation', 'all_relevant_sentence_keys', 'all_utilized_sentence_keys', 'trulens_groundedness', 'trulens_context_relevance', 'ragas_faithfulness', 'ragas_context_relevance', 'gpt3_adherence', 'gpt3_context_relevance', 'gpt35_utilization', 'relevance_score', 'utilization_score', 'completeness_score', 'question_ru', 'response_ru', 'documents_sentences_ru'],
         num_rows: 749
     })
     validation: Dataset({
         features: ['id', 'question', 'documents', 'response', 'generation_model_name', 'annotating_model_name', 'dataset_name', 'documents_sentences', 'response_sentences', 'sentence_support_information', 'unsupported_response_se

In [ ]:
DatasetDict(new_dataset['delucionqa']).push_to_hub(
    "CMCenjoyer/ragbench-ru",
    config_name="delucionqa",
    private=False,  
    commit_message="Add RAG-Bench Russian dataset with delucionqa"
)

ValueError: All datasets in `DatasetDict` should have the same features but features for 'validation' and 'test' don't match: {'id': Value(dtype='string', id=None), 'question': Value(dtype='string', id=None), 'documents': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None), 'response': Value(dtype='string', id=None), 'generation_model_name': Value(dtype='string', id=None), 'annotating_model_name': Value(dtype='string', id=None), 'dataset_name': Value(dtype='string', id=None), 'documents_sentences': Sequence(feature=Sequence(feature=Sequence(feature=Value(dtype='string', id=None), length=-1, id=None), length=-1, id=None), length=-1, id=None), 'response_sentences': Sequence(feature=Sequence(feature=Value(dtype='string', id=None), length=-1, id=None), length=-1, id=None), 'sentence_support_information': [{'explanation': Value(dtype='string', id=None), 'fully_supported': Value(dtype='bool', id=None), 'response_sentence_key': Value(dtype='string', id=None), 'supporting_sent

In [ ]:
from datasets import Dataset, Value, Sequence
def unify_dataset_features(dataset_dict):
    target_types = {
        'adherence_score': Value('float64'),
        'trulens_groundedness': Value('float64'),
        'trulens_context_relevance': Value('float64'),
        'ragas_faithfulness': Value('float64'),
        'ragas_context_relevance': Value('float64'),
        'gpt3_adherence': Value('float64'),
        'gpt3_context_relevance': Value('float64'),
        'gpt35_utilization': Value('float64'),
        'relevance_score': Value('float64'),
        'utilization_score': Value('float64'),
        'completeness_score': Value('float64')
    }
    
    unified_datasets = {}
    
    for split_name, dataset in dataset_dict.items():
        print(f"Обрабатываем {split_name}...")
        
        def convert_features(example):
            if isinstance(example.get('adherence_score'), bool):
                example['adherence_score'] = 1.0 if example['adherence_score'] else 0.0
            null_to_float_fields = [
                'trulens_groundedness', 'trulens_context_relevance', 
                'ragas_faithfulness', 'ragas_context_relevance',
                'gpt3_adherence', 'gpt3_context_relevance', 'gpt35_utilization'
            ]
            
            for field in null_to_float_fields:
                if example.get(field) is None:
                    example[field] = float('nan')
            
            return example
        converted_dataset = dataset.map(convert_features)
        features = converted_dataset.features.copy()
        for field, target_type in target_types.items():
            if field in features:
                features[field] = target_type
        
        converted_dataset = converted_dataset.cast(features)
        unified_datasets[split_name] = converted_dataset
    
    return DatasetDict(unified_datasets)

In [ ]:
def check_features_compatibility(dataset_dict):
    features_sets = {}
    
    for split_name, dataset in dataset_dict.items():
        features_sets[split_name] = set(str(feature) for feature in dataset.features.values())
    
    first_split = next(iter(features_sets))
    for split_name, features in features_sets.items():
        if features != features_sets[first_split]:
            print(f"❌ {split_name} отличается от {first_split}")
            diff = features.symmetric_difference(features_sets[first_split])
            for d in diff:
                print(f"   - {d}")
            return False
        else:
            print(f"✅ {split_name} совместим")
    
    return True

print("Проверка delucionqa:")
check_features_compatibility(DatasetDict(new_dataset['delucionqa']))

Проверка CUAD:
✅ train совместим
✅ validation совместим
❌ test отличается от train
   - Value(dtype='null', id=None)


False

In [47]:
fixed_delucionqa = unify_dataset_features(DatasetDict(new_dataset['delucionqa']).copy())

Обрабатываем train...


Casting the dataset: 100%|███████████████████████████████████████████████████████████████████████| 1458/1458 [00:00<00:00, 50382.65 examples/s]


Обрабатываем validation...


Casting the dataset: 100%|█████████████████████████████████████████████████████████████████████████| 182/182 [00:00<00:00, 19272.96 examples/s]


Обрабатываем test...


Casting the dataset: 100%|█████████████████████████████████████████████████████████████████████████| 184/184 [00:00<00:00, 18334.88 examples/s]


In [48]:
check_features_compatibility(fixed_delucionqa)

✅ train совместим
✅ validation совместим
✅ test совместим


True

In [49]:
fixed_delucionqa.push_to_hub(
    "CMCenjoyer/ragbench-ru",
    config_name="delucionqa",
    private=False,  # или False для публичного
    commit_message="Add RAG-Bench Russian dataset with delucionqa"
)

Uploading the dataset shards: 100%|██████████████████████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.67s/it]


CommitInfo(commit_url='https://huggingface.co/datasets/CMCenjoyer/ragbench-ru/commit/c9e21ee918789d73b0f439dfc3968d1f9cb21b12', commit_message='Add RAG-Bench Russian dataset with delucionqa', commit_description='', oid='c9e21ee918789d73b0f439dfc3968d1f9cb21b12', pr_url=None, pr_revision=None, pr_num=None)

In [50]:
fixed_cuad = unify_dataset_features(DatasetDict(new_dataset['cuad']).copy())

Обрабатываем train...


Casting the dataset: 100%|█████████████████████████████████████████████████████████████████████████| 749/749 [00:00<00:00, 21661.72 examples/s]


Обрабатываем validation...


Casting the dataset: 100%|█████████████████████████████████████████████████████████████████████████| 269/269 [00:00<00:00, 21249.98 examples/s]


Обрабатываем test...


Casting the dataset: 100%|█████████████████████████████████████████████████████████████████████████| 305/305 [00:00<00:00, 23698.83 examples/s]


In [51]:
fixed_cuad.push_to_hub(
    "CMCenjoyer/ragbench-ru",
    config_name="cuad",
    private=False,  # или False для публичного
    commit_message="Add RAG-Bench Russian dataset with cuad"
)

Uploading the dataset shards: 100%|██████████████████████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.99s/it]


CommitInfo(commit_url='https://huggingface.co/datasets/CMCenjoyer/ragbench-ru/commit/3c620febe1de26209af622f636f19008ee755e48', commit_message='Add RAG-Bench Russian dataset with cuad', commit_description='', oid='3c620febe1de26209af622f636f19008ee755e48', pr_url=None, pr_revision=None, pr_num=None)

In [ ]:
ds = load_dataset(
                "json",
                data_files={'train': 'checkpoints/emanual_train.jsonl'},
                split='train'
            )

In [63]:
import os
import json
from datasets import Dataset, DatasetDict, load_dataset
dataset_dir = 'checkpoints/'

files = [
    'covidqa_test.jsonl', 'covidqa_train.jsonl', 'covidqa_validation.jsonl',
    'cuad_test.jsonl', 'cuad_train.jsonl', 'cuad_validation.jsonl',
    'delucionqa_test.jsonl', 'delucionqa_train.jsonl', 'delucionqa_validation.jsonl',
    'expertqa_test.jsonl', 'expertqa_train.jsonl', 'expertqa_validation.jsonl',
    'finqa_test.jsonl', 'finqa_train.jsonl', 'finqa_validation.jsonl',
    'hagrid_test.jsonl', 'hagrid_train.jsonl', 'hagrid_validation.jsonl',
    'hotpotqa_test.jsonl', 'hotpotqa_train.jsonl', 'hotpotqa_validation.jsonl'
]
def load_jsonl_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        return [json.loads(line) for line in file]
import os
datasets = {}

# Загружаем данные для каждого датасета
for file in files:
    dataset_name = file.split('_')[0]  # Выделяем название датасета из имени файла
    file_path = os.path.join(dataset_dir, file)
    
    if dataset_name not in datasets:
        datasets[dataset_name] = {}

    # Читаем jsonl файл и добавляем его в соответствующий датасет
    datasets[dataset_name][file.split('.')[0].split('_')[1]] = load_dataset(
                "json",
                data_files={file.split('.')[0].split('_')[1]:file_path},
                split=file.split('.')[0].split('_')[1]
            )
    


Generating test split: 246 examples [00:00, 18189.81 examples/s]
Generating train split: 1252 examples [00:00, 23934.46 examples/s]
Generating validation split: 267 examples [00:00, 18812.65 examples/s]
Generating train split: 749 examples [00:00, 4176.36 examples/s]
Generating validation split: 269 examples [00:00, 4275.52 examples/s]
Generating train split: 1458 examples [00:00, 13418.33 examples/s]
Generating validation split: 182 examples [00:00, 11690.63 examples/s]
Generating test split: 188 examples [00:00, 8644.92 examples/s]
Generating train split: 1493 examples [00:00, 10884.90 examples/s]
Generating validation split: 196 examples [00:00, 9354.40 examples/s]
Generating test split: 2294 examples [00:00, 15611.28 examples/s]
Generating train split: 12502 examples [00:00, 15947.90 examples/s]
Generating validation split: 1766 examples [00:00, 15288.60 examples/s]
Generating test split: 1318 examples [00:00, 25782.93 examples/s]
Generating train split: 2892 examples [00:00, 27306

In [67]:
datasets.keys()

dict_keys(['covidqa', 'cuad', 'delucionqa', 'expertqa', 'finqa', 'hagrid', 'hotpotqa'])

In [72]:
datasets['finqa']

{'test': Dataset({
     features: ['id', 'question', 'documents', 'response', 'generation_model_name', 'annotating_model_name', 'dataset_name', 'documents_sentences', 'response_sentences', 'sentence_support_information', 'unsupported_response_sentence_keys', 'adherence_score', 'overall_supported_explanation', 'relevance_explanation', 'all_relevant_sentence_keys', 'all_utilized_sentence_keys', 'trulens_groundedness', 'trulens_context_relevance', 'ragas_faithfulness', 'ragas_context_relevance', 'gpt3_adherence', 'gpt3_context_relevance', 'gpt35_utilization', 'relevance_score', 'utilization_score', 'completeness_score', 'question_ru', 'response_ru', 'documents_sentences_ru'],
     num_rows: 2294
 }),
 'train': Dataset({
     features: ['id', 'question', 'documents', 'response', 'generation_model_name', 'annotating_model_name', 'dataset_name', 'documents_sentences', 'response_sentences', 'sentence_support_information', 'unsupported_response_sentence_keys', 'adherence_score', 'overall_suppo

In [71]:
# Создаем объект DatasetDict
for dataset_name, dataset_files in datasets.items():
    if dataset_name not in [ 'cuad', 'delucionqa']: 
        dataset_dict = unify_dataset_features(DatasetDict(dataset_files).copy())
        # Публикуем в Hugging Face Hub
        dataset_dict.push_to_hub(
            "CMCenjoyer/ragbench-ru", 
            config_name=dataset_name,
            private=False, 
            commit_message=f"Add RAG-Bench Russian dataset for {dataset_name}"
        )
        print(f"Dataset {dataset_name} uploaded successfully!")

Обрабатываем test...


Casting the dataset: 100%|█████████████████████████████████████████████████████████████████████████| 246/246 [00:00<00:00, 20403.38 examples/s]


Обрабатываем train...


Casting the dataset: 100%|███████████████████████████████████████████████████████████████████████| 1252/1252 [00:00<00:00, 47977.38 examples/s]


Обрабатываем validation...


...

Обрабатываем train...


Casting the dataset: 100%|███████████████████████████████████████████████████████████████████████| 1883/1883 [00:00<00:00, 68059.31 examples/s]


Обрабатываем validation...


Uploading the dataset shards: 100%|██████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.22it/s]


Dataset hotpotqa uploaded successfully!
